# DigiCom — Fine-tune DistilBERT (8-class dark-pattern classifier)

Trains `distilbert-base-uncased` from a **single CSV** (`Title,Category`), then exports +
**quantizes to int8 ONNX** (`model_quantized.onnx`, **opset 14** so the extension's ONNX
Runtime 1.14 can load it).

## Input — just one file
A CSV with columns **`Title,Category`** (Category = one of the 8 class strings).
On Colab it is read from Google Drive at:

```
MyDrive/dpbh enhance/dataset1.csv
```

**Nothing else is required.** The notebook does its own stratified **train / val / test**
split, so there's no separate eval file. It also **caps each class to ~200** rows so training
is balanced (down-samples over-represented classes like Not Dark Pattern; nothing is written
back to your CSV).

## Where to run
- **Recommended: Google Colab (T4 GPU)** — Runtime → Change runtime type → T4 GPU.
- **Alternative: Mac M4** — Python 3.11/3.12 venv + `pip install torch`; set `CSV_PATH` to the
  local file and skip the Drive-mount cell.


## 1. Install dependencies


In [ ]:
# Export to ONNX (opset 14) + dynamic int8 quantization — pure torch/onnxruntime, no optimum.
import torch
from pathlib import Path
from onnxruntime.quantization import quantize_dynamic, QuantType

ONNX_DIR = Path("onnx-export"); ONNX_DIR.mkdir(exist_ok=True)
fp32, quant = ONNX_DIR/"model.onnx", ONNX_DIR/"model_quantized.onnx"

# Wrap so the ONNX graph outputs raw logits (Transformers.js reads the "logits" output).
class LogitsOnly(torch.nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, input_ids, attention_mask):
        return self.m(input_ids=input_ids, attention_mask=attention_mask).logits

wrapped = LogitsOnly(model).eval().cpu()
ex = tok("example checkout text", return_tensors="pt")
torch.onnx.export(
    wrapped, (ex["input_ids"], ex["attention_mask"]), str(fp32),
    input_names=["input_ids", "attention_mask"], output_names=["logits"],
    dynamic_axes={"input_ids": {0: "batch", 1: "seq"},
                  "attention_mask": {0: "batch", 1: "seq"},
                  "logits": {0: "batch"}},
    opset_version=14, do_constant_folding=True)
print("fp32 ONNX:", fp32.stat().st_size // 1024, "KB")

# Dynamic int8 quantization (CPU) -> model_quantized.onnx (what the extension loads)
quantize_dynamic(str(fp32), str(quant), weight_type=QuantType.QInt8)
print("quantized ONNX:", quant.stat().st_size // 1024, "KB")
print("onnx files:", sorted(p.name for p in ONNX_DIR.glob("*.onnx")))

## 2. Config


In [ ]:
import os, json, numpy as np, torch
from pathlib import Path

# Keep label IDs IDENTICAL to the shipped model (extension/models/.../label_map.json)
LABELS = ["Forced Action","Misdirection","Not Dark Pattern","Obstruction",
          "Scarcity","Sneaking","Social Proof","Urgency"]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for i,l in enumerate(LABELS)}

# Drive path (Colab). On Mac, set this to your local CSV instead.
CSV_PATH = "/content/drive/MyDrive/dpbh enhance/dataset1.csv"

BASE_MODEL = "distilbert-base-uncased"
TARGET_PER_CLASS = 200       # cap each class to this many rows for a balanced trainset
SEED, EPOCHS, LR, BATCH, MAXLEN = 42, 8, 3e-5, 16, 128
OUT_DIR = Path("dpbh-distilbert-finetuned")

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

## 3. Mount Drive (Colab only — skip on Mac)


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Not on Colab (or already mounted):", e)
    print("Make sure CSV_PATH points to your local file.")
assert os.path.exists(CSV_PATH), f"CSV not found at {CSV_PATH} — fix CSV_PATH in cell 2."
print("using:", CSV_PATH)

## 4. Load CSV, validate, and balance (cap each class to TARGET_PER_CLASS)


In [ ]:
import pandas as pd
df = pd.read_csv(CSV_PATH)
assert {"Title","Category"}.issubset(df.columns), f"CSV must have Title,Category. Got {list(df.columns)}"
df = df[["Title","Category"]].dropna()
df["Title"] = df["Title"].astype(str).str.strip()
df["Category"] = df["Category"].astype(str).str.strip()
df = df[df["Title"].str.len() > 0]

# Drop rows whose Category is not one of the 8 known labels (report them)
unknown = sorted(set(df["Category"]) - set(LABELS))
if unknown: print("WARNING dropping unknown categories:", unknown)
df = df[df["Category"].isin(LABELS)]

# De-duplicate identical titles (case-insensitive)
df = df.assign(_key=df["Title"].str.lower()).drop_duplicates(subset="_key").drop(columns="_key")
print("rows after clean/dedup:", len(df))
print("raw per-class:\n", df["Category"].value_counts())

# Balance: cap each class to TARGET_PER_CLASS (down-sample; keep all if fewer)
parts = [g.sample(n=min(len(g), TARGET_PER_CLASS), random_state=SEED)
         for _, g in df.groupby("Category")]
bal = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
print("\nbalanced per-class:\n", bal["Category"].value_counts())
print("\nbalanced total:", len(bal))

## 5. Stratified train / val / test split (80 / 10 / 10)
The **test** split is the held-out set used for the final per-class report — no external file needed.


In [ ]:
from sklearn.model_selection import train_test_split
X = bal["Title"].tolist()
y = [label2id[c] for c in bal["Category"]]
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.20, random_state=SEED, stratify=y)
X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)
print("train:", len(X_tr), "| val:", len(X_va), "| test:", len(X_te))

## 6. Tokenize


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
def tokenize(b): return tok(b["text"], truncation=True, max_length=MAXLEN)
train_ds = Dataset.from_dict({"text":X_tr,"label":y_tr}).map(tokenize, batched=True)
val_ds   = Dataset.from_dict({"text":X_va,"label":y_va}).map(tokenize, batched=True)

## 7. Train


In [ ]:
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(LABELS), id2label=id2label, label2id=label2id)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "macro_f1": f1_score(p.label_ids, preds, average="macro", zero_division=0)}

args = TrainingArguments(output_dir="ckpt", num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
    eval_strategy="epoch", save_strategy="no", logging_steps=20, seed=SEED, report_to="none")

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    tokenizer=tok, data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
trainer.train()
print("validation:", trainer.evaluate())

## 8. Held-out test report (per-class precision / recall / F1)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
enc = tok(X_te, truncation=True, max_length=MAXLEN, padding=True, return_tensors="pt").to(model.device)
model.eval()
with torch.no_grad():
    pred = model(**enc).logits.argmax(-1).cpu().numpy()
print(classification_report(y_te, pred, target_names=LABELS, zero_division=0))
print("confusion (rows=true):\n", confusion_matrix(y_te, pred))

## 9. Save fp32 model + tokenizer + label_map.json


In [ ]:
OUT_DIR.mkdir(exist_ok=True)
model.save_pretrained(OUT_DIR); tok.save_pretrained(OUT_DIR)
json.dump({str(i): l for i, l in id2label.items()},
          open(OUT_DIR/"label_map.json","w"), ensure_ascii=False)
print("saved:", sorted(p.name for p in OUT_DIR.iterdir()))

## 10. Export to ONNX (opset 14) + dynamic int8 quantization


In [ ]:
from optimum.exporters.onnx import main_export
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

ONNX_DIR = Path("onnx-export")
main_export(model_name_or_path=str(OUT_DIR), output=str(ONNX_DIR),
            task="text-classification", opset=14)   # opset 14 -> loadable by ORT 1.14

quantizer = ORTQuantizer.from_pretrained(ONNX_DIR, file_name="model.onnx")
qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)  # dynamic int8
quantizer.quantize(save_dir=ONNX_DIR, quantization_config=qconfig)
print("onnx files:", sorted(p.name for p in ONNX_DIR.glob("*.onnx")))

## 11. Assemble the source-of-truth artifact (matches repo `onnx_quantized/`) + download


In [ ]:
import shutil
ART = Path("onnx_quantized_new")
ART.mkdir(exist_ok=True)
for f in ["config.json","tokenizer.json","tokenizer_config.json",
          "special_tokens_map.json","vocab.txt","label_map.json"]:
    if (OUT_DIR/f).exists(): shutil.copy(OUT_DIR/f, ART/f)
shutil.copy(ONNX_DIR/"model_quantized.onnx", ART/"model_quantized.onnx")
print("artifact files:", sorted(p.name for p in ART.iterdir()))

shutil.make_archive("onnx_quantized_new", "zip", ART)
try:
    from google.colab import files; files.download("onnx_quantized_new.zip")
except Exception:
    print("Local run — artifact dir:", ART.resolve())

## 12. Integrate into the extension (run in the repo, not the notebook)

```bash
# From repo root. Colab: unzip the downloaded onnx_quantized_new.zip first.
rm -f onnx_quantized/*
cp onnx_quantized_new/* onnx_quantized/

npm run sync-model          # copy into extension/models/dpbh-distilbert/ (onnx/ nested)
npm run sync-model:check    # verify the two copies match
npm run eval                # measure the NEW model on the held-out eval set
```

Then reload the extension (`chrome://extensions` → Remove → Load unpacked) → popup badge
should reach **ML: active**. Update `docs/MODEL_CARD.md` with the new numbers and bump the
version in `extension/manifest.json` + `CHANGELOG.md`.

> Extension runtime is `@xenova/transformers` v2 (ORT 1.14) loading `{ quantized: true }` →
> `onnx/model_quantized.onnx`, which is why we export at **opset 14**.
